In [25]:
# Imports 
import time
import requests
import pandas as pd
%pip install yfinance --quiet
import yfinance as yf
from pathlib import Path
import numpy as np
import warnings
import zipfile
from pathlib import Path

warnings.filterwarnings("ignore")

Note: you may need to restart the kernel to use updated packages.


In [26]:
'''
import os, subprocess

REPO_URL = "https://github.com/tongyuguo/HelpHerInvest.git"
REPO_DIR = "HelpHerInvest"
data_dir = os.path.join(REPO_DIR, "Data")

def clone_or_pull():
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "pull"])
    else:
        subprocess.run(["git", "clone", REPO_URL])

clone_or_pull()
'''


'\nimport os, subprocess\n\nREPO_URL = "https://github.com/tongyuguo/HelpHerInvest.git"\nREPO_DIR = "HelpHerInvest"\ndata_dir = os.path.join(REPO_DIR, "Data")\n\ndef clone_or_pull():\n    if os.path.isdir(os.path.join(REPO_DIR, ".git")):\n        subprocess.run(["git", "-C", REPO_DIR, "pull"])\n    else:\n        subprocess.run(["git", "clone", REPO_URL])\n\nclone_or_pull()\n'

In [27]:
'''
# X-Variables #
DATA_DIR = Path(REPO_DIR) / "Data"
## Create Path to Dataset ##
dataset_path = f"{REPO_DIR}/Data/stock_symbols_new.csv.zip"
csv_file_name = "stock_symbols_new.csv"
'''

dataset_path = "/Users/christoomey/Desktop/Courses/ADAN8888_Applied_Analytics_Project/Data/stock_symbols_new.csv.zip"
csv_file_name = "stock_symbols_new.csv"


# Open the zip file and then open the specific CSV file within it
with zipfile.ZipFile(dataset_path, 'r') as zf:
    with zf.open(csv_file_name) as file_handle:
        # Read the file-like object directly into pandas
        symbols = pd.read_csv(file_handle)

## Reading Symbols from Dataset ##
tickers = symbols['symbol'].tolist()
#tickers = tickers[:50]  # limit to 50 for testing
original_tickers = tickers 

print(len(original_tickers))

all_tickers = [item for item in original_tickers if isinstance(item, str)]

print(len(all_tickers[:2000]))

all_tickers = all_tickers[:2000]  # limit to 2000 for testing


## Download Price Data ##

def download_adj_close_in_batches(
    tickers,
    start="2010-01-01",
    batch_size=100,
    max_retries=4,
    base_sleep=2,
    threads=False,   # often more stable than True for large jobs
):
    all_prices = []
    failed = []

    # de-dupe while preserving order
    tickers = list(dict.fromkeys(tickers))

    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i + batch_size]
        print(f"Batch {i//batch_size + 1}: {len(batch)} tickers")

        success = False
        for attempt in range(1, max_retries + 1):
            try:
                df = yf.download(
                    batch,
                    start=start,
                    auto_adjust=False,
                    progress=False,
                    threads=threads,
                )

                # Handle both single and multi-ticker responses
                if isinstance(df.columns, pd.MultiIndex):
                    adj = df["Adj Close"]
                else:
                    # single ticker edge case
                    adj = df.rename(columns={"Adj Close": batch[0]})[batch[0]].to_frame()

                all_prices.append(adj)
                success = True
                break

            except Exception as e:
                wait = base_sleep * (2 ** (attempt - 1)) + random.uniform(0, 1)
                print(f"  Attempt {attempt}/{max_retries} failed: {e}. Retrying in {wait:.1f}s")
                time.sleep(wait)

        if not success:
            failed.extend(batch)

        # small pause between batches to reduce API stress
        time.sleep(1)

    prices = pd.concat(all_prices, axis=1) if all_prices else pd.DataFrame()
    prices = prices.loc[:, ~prices.columns.duplicated()]  # guard against duplicates
    prices = prices.sort_index(axis=1)

    return prices, failed

prices, failed = download_adj_close_in_batches(all_tickers, batch_size=100)

print("Prices shape:", prices.shape)
print("Failed tickers:", failed[:20], "..." if len(failed) > 20 else "")

prices = prices.dropna(how='all')

print(prices.head(10))

## Compute Features ##
monthly_px = prices.resample('ME').last()  # month-end prices
mom_1m  = monthly_px / monthly_px.shift(1)  - 1  # 1-month momentum
mom_3m  = monthly_px / monthly_px.shift(3)  - 1  # 3-month momentum
mom_6m  = monthly_px / monthly_px.shift(6)  - 1  # 6-month momentum
mom_12m = monthly_px / monthly_px.shift(12) - 1  # 12-month momentum
mom_12m_ex_1m = (monthly_px.shift(1) / monthly_px.shift(12)) - 1  # 12-month momentum excluding most recent month

rel_3m_spy  = mom_3m.sub(mom_3m["SPY"], axis=0)  # relative strength against S&P 3-month
rel_6m_spy  = mom_6m.sub(mom_6m["SPY"], axis=0)  # relative strength against S&P 6-month
rel_12m_spy = mom_12m.sub(mom_12m["SPY"], axis=0) # relative strength against S&P 12-month

daily_ret = prices.pct_change() # daily returns

vol_3m = (daily_ret.rolling(63).std() * np.sqrt(252)).resample("M").last() # 3-month volatility
vol_6m = (daily_ret.rolling(126).std() * np.sqrt(252)).resample("M").last() # 6-month volatility


roll_max_6m  = monthly_px.rolling(6).max()  # 6-month rolling max
roll_max_12m = monthly_px.rolling(12).max() # 12-month rolling max

drawdown_6m  = monthly_px / roll_max_6m  - 1  # 6-month drawdown
drawdown_12m = monthly_px / roll_max_12m - 1  # 12-month drawdown

dma_200 = prices.rolling(200).mean().resample("M").last() # 200-day moving average
pct_above_200dma = monthly_px / dma_200 - 1  # pct above 200-day moving average


## Combine Features ##
X = pd.concat(
    {
        "mom_1m": mom_1m[all_tickers],
        "mom_3m": mom_3m[all_tickers],
        "mom_6m": mom_6m[all_tickers],
        "mom_12m": mom_12m[all_tickers],
        "mom_12m_ex_1m": mom_12m_ex_1m[all_tickers],
        "rel_3m_spy": rel_3m_spy[all_tickers],
        "rel_6m_spy": rel_6m_spy[all_tickers],
        "rel_12m_spy": rel_12m_spy[all_tickers],
        "vol_3m": vol_3m[all_tickers],
        "vol_6m": vol_6m[all_tickers],
        #"downside_vol_6m": downside_vol_6m[all_tickers],
        "drawdown_6m": drawdown_6m[all_tickers],
        "drawdown_12m": drawdown_12m[all_tickers],
        "pct_above_200dma": pct_above_200dma[all_tickers],
    },
    axis=1
)


## Standardize Data Function - z score ##
def zscore_cs(row: pd.Series) -> pd.Series:
    # row contains values across tickers for a single feature at a single date
    mu = row.mean()
    sd = row.std(ddof=0)
    if sd == 0 or np.isnan(sd):
        return row * 0.0
    return (row - mu) / sd # calcs z-score


## Normalize per feature across tickers at each date ##
X_z = X.copy()
#print("X_z before normalization:")
#print(X_z.head(20))

for feat in X.columns.get_level_values(0).unique():
    X_z[feat] = X[feat].apply(zscore_cs, axis=1)

#print("X_z after normalization:")
#print(X_z.head(20))

## Flatten X_z table so each  ticker is a row ##
X_panel = (
    X_z.stack(level=1)              # index becomes (Date, Ticker)
      .rename_axis(index=["Date","Ticker"])
      .reset_index()
)


#print(X_panel.head())
print("-----  -----  -----")
print(X_panel.tail())
print("-----  -----  -----")

print("Size of dataset:",
"Rows:",X_panel.shape[0],
"Columns:",X_panel.shape[1])  


#output = DATA_DIR / "dependent_variables.csv"

##
## <<< this output needs to be change >>> ##
##
output = "/Users/christoomey/Desktop/Courses/ADAN8888_Applied_Analytics_Project/Data/test_dataset.csv"
X_panel.to_csv(output, index=False)
print(f"Output file saved to: {output}")


10284
2000
Batch 1: 100 tickers
Batch 2: 100 tickers


$UTX: possibly delisted; no timezone found

1 Failed download:
['UTX']: possibly delisted; no timezone found


Batch 3: 100 tickers
Batch 4: 100 tickers
Batch 5: 100 tickers
Batch 6: 100 tickers
Batch 7: 100 tickers
Batch 8: 100 tickers
Batch 9: 100 tickers
Batch 10: 100 tickers
Batch 11: 100 tickers
Batch 12: 100 tickers
Batch 13: 100 tickers
Batch 14: 100 tickers
Batch 15: 100 tickers
Batch 16: 100 tickers
Batch 17: 100 tickers
Batch 18: 100 tickers


$SAND: possibly delisted; no timezone found
$WNS: possibly delisted; no timezone found

2 Failed downloads:
['SAND', 'WNS']: possibly delisted; no timezone found


Batch 19: 100 tickers
Batch 20: 100 tickers
Prices shape: (4060, 2000)
Failed tickers: [] 
Ticker              A         AA       AAL      AAON        AAP  AAPG  \
Date                                                                    
2010-01-04  19.856192  35.359665  4.496876  3.467879  34.742352   NaN   
2010-01-05  19.640499  34.255352  5.005957  3.367261  34.535847   NaN   
2010-01-06  19.570724  36.039246  4.798553  3.233680  34.836990   NaN   
2010-01-07  19.545336  35.274719  4.939965  3.353383  34.828388   NaN   
2010-01-08  19.539000  36.145443  4.845691  3.393283  34.966042   NaN   
2010-01-11  19.551687  37.058609  4.751417  3.485228  34.621902   NaN   
2010-01-12  19.316965  32.959881  4.789126  3.434918  34.019619   NaN   
2010-01-13  19.469202  33.936783  5.166224  3.457471  34.492825   NaN   
2010-01-14  19.761032  33.575752  5.269925  3.454002  34.036831   NaN   
2010-01-15  19.304277  33.193485  5.185078  3.322155  33.821743   NaN   

Ticker          AAPL  AAUC      

In [28]:
# Y-Variable #
import pandas as pd
import yfinance as yf

#df_path = "/Users/christoomey/Desktop/Courses/ADAN8888_Applied_Analytics_Project/Data/final_dataset_20260224.csv"
#df = pd.read_csv(df_path)
#all_tickers = df["Ticker"].unique().tolist()
#print(len(all_tickers), "unique tickers in dataset")

def forward_excess_return_monthly(tickers, benchmark="SPY", start="2010-01-01", end=None, horizon_months=3):

    universe = list(dict.fromkeys(list(tickers) + [benchmark]))

    px_daily = yf.download(
        universe, start=start, end=end, auto_adjust=False, progress=False
    )["Adj Close"]

    px_m = px_daily.resample("M").last().dropna(subset=[benchmark])

    fwd_ret = px_m.shift(-horizon_months) / px_m - 1.0
    bench_fwd = fwd_ret[benchmark].rename("bench_fwd_return")

    # Wide to long in one go
    out = pd.DataFrame(index=px_m.index)
    out["bench_fwd_return"] = bench_fwd

    for t in tickers:
        out[f"adj_close_{t}"] = px_m[t]
        out[f"fwd_return_{t}"] = fwd_ret[t]
        out[f"fwd_excess_{t}"] = fwd_ret[t] - bench_fwd

    return out

df_y_m = forward_excess_return_monthly(all_tickers, benchmark="SPY", start="2010-01-01", horizon_months=3)
print(df_y_m.head(10))
print(df_y_m.tail(10))

bench_df = (
    df_y_m[["bench_fwd_return"]]
    .rename(columns={"bench_fwd_return": "bench_fwd_return"})
    .reset_index()
)

# Keep only stock-level columns
stock_cols = [c for c in df_y_m.columns if "_" in c and not c.startswith("fwd_ret_bench")]

y_long = (
    df_y_m[stock_cols]
    .reset_index()
    .melt(id_vars="Date", var_name="metric_ticker", value_name="value")
)

# Split "adj_close_AAPL" -> metric="adj_close", ticker="AAPL"
y_long[["metric", "Ticker"]] = y_long["metric_ticker"].str.rsplit("_", n=1, expand=True)

y_long = y_long.drop(columns="metric_ticker")

df_y_m_output = (
    y_long
    .pivot(index=["Date", "Ticker"], columns="metric", values="value")
    .reset_index()
)

df_y_m_output = df_y_m_output[["Date", "Ticker", "adj_close", "fwd_excess", "fwd_return"]]
df_y_m_output = df_y_m_output[df_y_m_output["Ticker"] != "return"]

print(df_y_m_output.head(10))
print(df_y_m_output.tail(10))

print("Size of dataset:",
"Rows:",df_y_m_output.shape[0],
"Columns:",df_y_m_output.shape[1])  

# Save to CSV
#y_output = DATA_DIR / "independent_variables.csv"
y_output = "/Users/christoomey/Desktop/Courses/ADAN8888_Applied_Analytics_Project/independent_variables.csv"
df_y_m_output.to_csv(y_output, index=False)
print(f"Output file saved to: {y_output}")


$UTX: possibly delisted; no timezone found
$WNS: possibly delisted; no timezone found
$SAND: possibly delisted; no timezone found

3 Failed downloads:
['UTX', 'WNS', 'SAND']: possibly delisted; no timezone found


            bench_fwd_return  adj_close_NVDA  fwd_return_NVDA  \
Date                                                            
2010-01-31          0.110897        0.352752         0.020793   
2010-02-28         -0.008304        0.371318        -0.188889   
2010-03-31         -0.113580        0.398823        -0.413218   
2010-04-30         -0.067464        0.360087        -0.415022   
2010-05-31         -0.032540        0.301180        -0.289954   
2010-06-30          0.111616        0.234022         0.143977   
2010-07-31          0.080297        0.210643         0.307944   
2010-08-31          0.131178        0.213852         0.458735   
2010-09-30          0.107608        0.267716         0.318493   
2010-10-31          0.091710        0.275509         0.990016   

            fwd_excess_NVDA  adj_close_GOOGL  fwd_return_GOOGL  \
Date                                                             
2010-01-31        -0.090104        13.162311         -0.008001   
2010-02-28        -0.

In [29]:


# Check first
print(X_panel.dtypes[["Date", "Ticker"]])
print(df_y_m_output.dtypes[["Date", "Ticker"]])

# Fix Date type on both sides
X_panel["Date"] = pd.to_datetime(X_panel["Date"], errors="coerce")
df_y_m_output["Date"] = pd.to_datetime(df_y_m_output["Date"], errors="coerce")


merged_df = pd.merge(
    X_panel,
    df_y_m_output,
    on=["Date", "Ticker"],
    how="inner"
)

merged_df = merged_df.round(4)

print("Size of merged dataset:",
"Rows:",merged_df.shape[0],
"Columns:",merged_df.shape[1])

final_dataset_path = "/Users/christoomey/Desktop/Courses/ADAN8888_Applied_Analytics_Project/Data/final_dataset_20260224v2.csv"
merged_df.to_csv(final_dataset_path, index=False)
print(f"Final dataset saved to: {final_dataset_path}")


Date      datetime64[ns]
Ticker            object
dtype: object
metric
Date      datetime64[ns]
Ticker            object
dtype: object
Size of merged dataset: Rows: 302024 Columns: 18
Final dataset saved to: /Users/christoomey/Desktop/Courses/ADAN8888_Applied_Analytics_Project/Data/final_dataset_20260224v2.csv
